# Consultas

**Responsable:** Juan Hernandez

**Objetivo:** consultas 1–9, 12–15 y las dos propias. Cada una con: pregunta, código, resultado e interpretación.
Las consultas 10 y 11 (bibliotecas `.fasta`) están en `05_bibliotecas_y_figuras.ipynb`.

Modelo de referencia: `../MODELO.md`. Base: `panteon` restaurada del dump (`../resultados/dump_info.md`).

**Convenciones:**

- Toda consulta se resuelve en MongoDB (`find` o `aggregate`); pandas solo se usa para mostrar el resultado.
- Las agregaciones por clasificación del TE usan `classification.te_class`, `te_order` y `te_superfamily`; las
  de taxonomía de la especie usan `kingdom`, `phylum` (copiados en `tes`) o `taxonomy.*` (en `especies`).
- Para filtrar por especie se usa `species_id`, resuelto desde el nombre con `normalize_species`, porque
  `tes.species_name` conserva el nombre del FASTA (4 especies difieren del CSV por `subsp.`/`sp.`).
- `gc_content` y `n_percent` se guardan como fracción; aquí se muestran en porcentaje.

## 0. Conexión y parámetros

In [ ]:
import json
import sys
import time

import pandas as pd
from pymongo import MongoClient

sys.path.insert(0, "..")
from src.normalizacion import normalize_species

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

URI = "mongodb://localhost:27017"
client = MongoClient(URI)
db = client["panteon"]
tes, especies, kmers = db.tes, db.especies, db.kmers

# Parámetros de las consultas que reciben una especie, un TE o un k (se pueden cambiar)
ESPECIE = "Halicephalobus mephisto"   # nombre del CSV o del FASTA (con o sin puntos)
SUPERFAMILIA = "HELITRON"
PDB_ID = "PDB00000001"
K = 2

### Verificación de la base

Conteos e índices esperados según `MODELO.md`. Si algo no coincide, la base no es la restaurada del dump.

In [ ]:
ESPERADO = {"especies": 2790, "tes": 239946, "kmers": 1439676}
INDICES = {
    "tes": [[("pdb_id", 1)], [("species_id", 1)], [("species_name", 1), ("classification.te_superfamily", 1)], [("kingdom", 1)]],
    "kmers": [[("te_id", 1), ("k", 1)], [("pdb_id", 1), ("k", 1)], [("k", 1), ("te_order", 1)]],
    "especies": [[("species", 1)]],
}

print("MongoDB", client.server_info()["version"])
for nombre, esperado in ESPERADO.items():
    n = db[nombre].estimated_document_count()
    marca = "OK" if n == esperado else f"ATENCIÓN: se esperaban {esperado:,}"
    print(f"{nombre:9s} {n:>10,} documentos  {marca}")
for nombre, esperados in INDICES.items():
    existentes = [[(k, int(v)) for k, v in i["key"].items()] for i in db[nombre].list_indexes()]
    faltan = [e for e in esperados if e not in existentes]
    assert not faltan, f"Faltan índices en {nombre}: {faltan}"
print("Índices de MODELO.md presentes")

### Funciones auxiliares

- `a_tabla`: pasa la salida de un `$group` a un `DataFrame`, abriendo el `_id` compuesto en columnas.
- `resolver_especie`: busca una especie por nombre exacto (índice `{species: 1}`) y, si no la encuentra,
  compara con `normalize_species`, así acepta tanto `Beta vulgaris subsp vulgaris` (CSV) como
  `Beta vulgaris subsp. vulgaris` (FASTA).
- `resumen_explain`: ejecuta `explain` con `executionStats` y resume el plan, para mostrar qué índice usa una consulta.

In [ ]:
def a_tabla(filas, id_como=None):
    """DataFrame a partir de documentos; abre los subdocumentos (_id.x -> x)."""
    df = pd.json_normalize(list(filas))
    if df.empty:
        return df
    de_id = [c for c in df.columns if c.startswith("_id.")]
    df = df[de_id + [c for c in df.columns if c not in de_id]]   # columnas del _id primero
    df.columns = [c.removeprefix("_id.") for c in df.columns]
    if id_como and "_id" in df.columns:
        df = df.rename(columns={"_id": id_como})
    return df


def resolver_especie(nombre):
    """Documento de `especies` (con _id, species y taxonomy) cuyo nombre coincide con `nombre`."""
    doc = especies.find_one({"species": nombre.strip()})
    if doc:
        return doc
    clave = normalize_species(nombre)
    candidatos = [e for e in especies.find() if normalize_species(e["species"]) == clave]
    if not candidatos:
        raise ValueError(f"No hay ninguna especie llamada {nombre!r}")
    return candidatos[0]


def _etapas(plan):
    salida = []
    while isinstance(plan, dict):
        if "stage" in plan:
            salida.append(plan["stage"] + (f" {plan['indexName']}" if "indexName" in plan else ""))
        plan = plan.get("queryPlan") or plan.get("inputStage") or (plan.get("inputStages") or [None])[0]
    return salida


def resumen_explain(comando):
    """Resumen de explain(executionStats) para un comando find."""
    salida = db.command("explain", comando, verbosity="executionStats")
    stats = salida["executionStats"]
    return {
        "plan": " > ".join(_etapas(salida["queryPlanner"]["winningPlan"])),
        "claves_examinadas": stats["totalKeysExamined"],
        "docs_examinados": stats["totalDocsExamined"],
        "devueltos": stats["nReturned"],
        "tiempo_ms": stats["executionTimeMillis"],
    }


def medir(funcion):
    """Ejecuta funcion() y devuelve (resultado, segundos)."""
    t0 = time.perf_counter()
    r = funcion()
    return r, time.perf_counter() - t0

## Cantidad de elementos

### Consulta 1

**Pregunta:** ¿Cuántos TEs hay por clase, por orden y por superfamilia en toda la base?

Un solo `aggregate` con `$facet` calcula los tres niveles en una pasada sobre `tes`. El orden se agrupa junto
con su clase y la superfamilia junto con su clase y orden, porque un mismo nombre de superfamilia puede
repetir el del orden (por ejemplo `LTR/LTR`). No usa índices: recorre toda la colección (COLLSCAN), lo
esperado en un conteo global.

In [ ]:
pipeline_q1 = [
    {"$facet": {
        "clase": [
            {"$group": {"_id": "$classification.te_class", "n_tes": {"$sum": 1}}},
            {"$sort": {"n_tes": -1}},
        ],
        "orden": [
            {"$group": {"_id": {"clase": "$classification.te_class", "orden": "$classification.te_order"},
                        "n_tes": {"$sum": 1}}},
            {"$sort": {"n_tes": -1}},
        ],
        "superfamilia": [
            {"$group": {"_id": {"clase": "$classification.te_class", "orden": "$classification.te_order",
                                "superfamilia": "$classification.te_superfamily"},
                        "n_tes": {"$sum": 1}}},
            {"$sort": {"n_tes": -1}},
        ],
    }},
]

q1, t = medir(lambda: next(tes.aggregate(pipeline_q1)))
total = sum(d["n_tes"] for d in q1["clase"])
q1_tablas = {}
for nivel, filas in q1.items():
    df = a_tabla(filas, id_como="clase")
    df["porcentaje"] = (100 * df["n_tes"] / total).round(2)
    q1_tablas[nivel] = df
print(f"Total: {total:,} TEs | {len(q1_tablas['orden'])} órdenes | "
      f"{len(q1_tablas['superfamilia'])} superfamilias | {t:.2f} s")
q1_tablas["clase"]

In [ ]:
q1_tablas["orden"]

In [ ]:
q1_tablas["superfamilia"]

**Interpretación:** _pendiente; se escribe con los resultados de la base completa._

### Consulta 2

**Pregunta:** ¿Cuántos TEs tiene cada especie, desglosados por clase, orden y superfamilia?

Se agrupa por `species_id` y los tres niveles de clasificación. `species_name`, `kingdom` y `source` salen
con `$first` del mismo documento de `tes` (son copias de la especie, decisión 1 del modelo), así que no hace
falta `$lookup`. El resultado completo es una tabla larga (una fila por especie y superfamilia) que se guarda
en `../resultados/consulta02_tes_por_especie.csv`; abajo se muestran un resumen por especie y el desglose
de la especie del parámetro `ESPECIE`.

In [ ]:
pipeline_q2 = [
    {"$group": {
        "_id": {"species_id": "$species_id", "clase": "$classification.te_class",
                "orden": "$classification.te_order", "superfamilia": "$classification.te_superfamily"},
        "especie": {"$first": "$species_name"},
        "reino": {"$first": "$kingdom"},
        "n_tes": {"$sum": 1},
    }},
    {"$sort": {"especie": 1, "_id.clase": 1, "_id.orden": 1, "_id.superfamilia": 1}},
]

filas_q2, t = medir(lambda: list(tes.aggregate(pipeline_q2, allowDiskUse=True)))
q2 = a_tabla(filas_q2)[["species_id", "especie", "reino", "clase", "orden", "superfamilia", "n_tes"]]
q2.to_csv("../resultados/consulta02_tes_por_especie.csv", index=False)
print(f"{len(q2):,} filas (especie x superfamilia) de {q2['species_id'].nunique():,} especies | {t:.2f} s")

# Resumen: una fila por especie, una columna por orden de TE
resumen_q2 = (q2.pivot_table(index=["especie", "reino"], columns="orden", values="n_tes", aggfunc="sum", fill_value=0))
resumen_q2.insert(0, "total", resumen_q2.sum(axis=1))
resumen_q2 = resumen_q2.sort_values("total", ascending=False)
resumen_q2.head(15)

Desglose de una especie. La misma agregación, con un `$match` por `species_id` al inicio (usa el índice
`{species_id: 1}`), sirve para consultar cualquier especie sin calcular las 2.790.

In [ ]:
def desglose_especie(nombre):
    esp = resolver_especie(nombre)
    filas = tes.aggregate([{"$match": {"species_id": esp["_id"]}}] + pipeline_q2)
    return esp, a_tabla(filas)[["clase", "orden", "superfamilia", "n_tes"]]


esp, df = desglose_especie(ESPECIE)
print(f"{esp['species']} ({esp['taxonomy']['kingdom']}): {df['n_tes'].sum():,} TEs")
df

**Interpretación:** _pendiente; se escribe con los resultados de la base completa._

### Consulta 3

**Pregunta:** ¿Cómo cambia la cantidad de TEs por clase y orden entre reinos (Animalia, Plantae, Fungi)?
¿Y entre phyla?

`kingdom` y `phylum` están copiados en `tes`, así que basta un `$group`. Un segundo `$group` calcula el total
de cada reino para expresar cada orden como porcentaje del reino: los reinos tienen distinto número de
especies, y comparar conteos absolutos confundiría la composición con el tamaño de la muestra.

In [ ]:
def composicion_por(campo, etiqueta):
    """Conteo y porcentaje de TEs por (campo, clase, orden), con el total de cada grupo."""
    return [
        {"$group": {"_id": {etiqueta: {"$ifNull": [campo, "(sin dato)"]},
                            "clase": "$classification.te_class", "orden": "$classification.te_order"},
                    "n_tes": {"$sum": 1}}},
        {"$group": {"_id": f"$_id.{etiqueta}", "total": {"$sum": "$n_tes"},
                    "filas": {"$push": {"clase": "$_id.clase", "orden": "$_id.orden", "n_tes": "$n_tes"}}}},
        {"$unwind": "$filas"},
        {"$project": {"_id": 0, etiqueta: "$_id", "clase": "$filas.clase", "orden": "$filas.orden",
                      "n_tes": "$filas.n_tes", "total_grupo": "$total",
                      "porcentaje": {"$round": [{"$multiply": [100, {"$divide": ["$filas.n_tes", "$total"]}]}, 2]}}},
        {"$sort": {etiqueta: 1, "n_tes": -1}},
    ]


q3_reino = a_tabla(tes.aggregate(composicion_por("$kingdom", "reino")))

# Por clase: conteo y porcentaje dentro de cada reino
por_clase = q3_reino.groupby(["reino", "clase"])["n_tes"].sum().unstack("reino")
display((100 * por_clase / por_clase.sum()).round(2).rename(columns=lambda c: f"% {c}").join(por_clase))

In [ ]:
# Por orden: porcentaje de cada orden dentro de cada reino
q3_reino.pivot_table(index=["clase", "orden"], columns="reino", values="porcentaje", fill_value=0)

In [ ]:
q3_reino.pivot_table(index=["clase", "orden"], columns="reino", values="n_tes", fill_value=0, aggfunc="sum")

Entre phyla. Se agrupa además por reino para no mezclar phyla de reinos distintos, y se cuenta cuántas
especies aporta cada phylum: un phylum con pocas especies da porcentajes poco estables.

In [ ]:
pipeline_phyla = [
    {"$group": {"_id": {"reino": "$kingdom", "phylum": {"$ifNull": ["$phylum", "(sin dato)"]}},
                "n_tes": {"$sum": 1}, "especies": {"$addToSet": "$species_id"},
                "n_ltr": {"$sum": {"$cond": [{"$eq": ["$classification.te_order", "LTR"]}, 1, 0]}},
                "n_clase_i": {"$sum": {"$cond": [{"$eq": ["$classification.te_class", "CLASSI"]}, 1, 0]}}}},
    {"$project": {"_id": 0, "reino": "$_id.reino", "phylum": "$_id.phylum", "n_especies": {"$size": "$especies"},
                  "n_tes": 1,
                  "tes_por_especie": {"$round": [{"$divide": ["$n_tes", {"$size": "$especies"}]}, 1]},
                  "pct_clase_i": {"$round": [{"$multiply": [100, {"$divide": ["$n_clase_i", "$n_tes"]}]}, 1]},
                  "pct_ltr": {"$round": [{"$multiply": [100, {"$divide": ["$n_ltr", "$n_tes"]}]}, 1]}}},
    {"$sort": {"reino": 1, "n_tes": -1}},
]
q3_phyla_resumen = a_tabla(tes.aggregate(pipeline_phyla))
q3_phyla_resumen

In [ ]:
# Porcentaje de cada orden en los phyla con al menos 10 especies
q3_phylum = a_tabla(tes.aggregate(composicion_por("$phylum", "phylum")))
phyla_grandes = q3_phyla_resumen.loc[q3_phyla_resumen["n_especies"] >= 10, "phylum"]
(q3_phylum[q3_phylum["phylum"].isin(phyla_grandes)]
 .pivot_table(index="phylum", columns="orden", values="porcentaje", fill_value=0))

**Interpretación:** _pendiente; se escribe con los resultados de la base completa._

### Consulta 4

**Pregunta:** ¿Cuáles son las 10 especies con más TEs y las 10 con menos?

Se cuentan los TEs por `species_id` y un `$facet` ordena en ambos sentidos. Para desempatar se usa el nombre
(orden alfabético), y una tercera rama cuenta cuántas especies hay con cada número de TEs, para saber si el
corte del top 10 deja especies empatadas por fuera. Solo a las 20 especies del resultado se les hace
`$lookup` a `especies`, para marcar las entradas que no son especies (`is_species: false`, por ejemplo
*Homininae*).

In [ ]:
marcar = [
    {"$lookup": {"from": "especies", "localField": "_id", "foreignField": "_id", "as": "esp"}},
    {"$project": {"_id": 0, "especie": 1, "reino": 1, "fuente": 1, "n_tes": 1,
                  "es_especie": {"$first": "$esp.is_species"}}},
]
pipeline_q4 = [
    {"$group": {"_id": "$species_id", "especie": {"$first": "$species_name"}, "reino": {"$first": "$kingdom"},
                "fuente": {"$first": "$source"}, "n_tes": {"$sum": 1}}},
    {"$facet": {
        "mas": [{"$sort": {"n_tes": -1, "especie": 1}}, {"$limit": 10}] + marcar,
        "menos": [{"$sort": {"n_tes": 1, "especie": 1}}, {"$limit": 10}] + marcar,
        "por_conteo": [{"$group": {"_id": "$n_tes", "n_especies": {"$sum": 1}}}],
    }},
]
q4 = next(tes.aggregate(pipeline_q4))
mas, menos = pd.DataFrame(q4["mas"]), pd.DataFrame(q4["menos"])
por_conteo = {d["_id"]: d["n_especies"] for d in q4["por_conteo"]}

for nombre, df in [("más", mas), ("menos", menos)]:
    corte = df["n_tes"].iloc[-1]
    print(f"Top 10 con {nombre} TEs: el corte es {corte} TEs; hay {por_conteo[corte]} especies con exactamente ese número")
mas.index = menos.index = range(1, 11)
display(mas)
menos

**Interpretación:** _pendiente; se escribe con los resultados de la base completa._

## Longitud y contenido GC

### Consulta 5

**Pregunta:** ¿Cuál es la longitud promedio, mínima y máxima de los TEs por clase, orden y superfamilia?

Además de lo pedido se calcula la mediana (`$median`, disponible desde MongoDB 7.0) y la desviación
estándar: las longitudes de TEs suelen tener colas largas, y la mediana muestra si el promedio está
arrastrado por unos pocos elementos muy largos.

In [ ]:
def estadisticas_longitud(id_grupo):
    return [
        {"$group": {"_id": id_grupo, "n_tes": {"$sum": 1},
                    "promedio": {"$avg": "$length"},
                    "mediana": {"$median": {"input": "$length", "method": "approximate"}},
                    "de": {"$stdDevSamp": "$length"},
                    "minimo": {"$min": "$length"}, "maximo": {"$max": "$length"}}},
        {"$set": {"promedio": {"$round": ["$promedio", 1]}, "de": {"$round": ["$de", 1]}}},
        {"$sort": {"promedio": -1}},
    ]


pipeline_q5 = [{"$facet": {
    "clase": estadisticas_longitud("$classification.te_class"),
    "orden": estadisticas_longitud({"clase": "$classification.te_class", "orden": "$classification.te_order"}),
    "superfamilia": estadisticas_longitud({"clase": "$classification.te_class", "orden": "$classification.te_order",
                                           "superfamilia": "$classification.te_superfamily"}),
}}]
q5 = {nivel: a_tabla(filas, id_como="clase") for nivel, filas in next(tes.aggregate(pipeline_q5)).items()}
q5["clase"]

In [ ]:
q5["orden"]

In [ ]:
q5["superfamilia"]

**Interpretación:** _pendiente; se escribe con los resultados de la base completa._

### Consulta 6

**Pregunta:** ¿Cuál es el porcentaje de GC promedio por orden y por reino?

`gc_content` se calculó por TE sin contar las N en el denominador, así que el promedio es el promedio de los
porcentajes de cada TE (cada TE pesa lo mismo, sin importar su longitud). También se cruza orden × reino,
porque un mismo orden puede tener GC distinto según el genoma del que viene.

In [ ]:
def gc_por(id_grupo):
    return [
        {"$group": {"_id": id_grupo, "n_tes": {"$sum": 1},
                    "gc_promedio": {"$avg": "$gc_content"}, "gc_de": {"$stdDevSamp": "$gc_content"},
                    "gc_mediana": {"$median": {"input": "$gc_content", "method": "approximate"}}}},
        {"$set": {"gc_promedio": {"$round": [{"$multiply": [100, "$gc_promedio"]}, 2]},
                  "gc_de": {"$round": [{"$multiply": [100, "$gc_de"]}, 2]},
                  "gc_mediana": {"$round": [{"$multiply": [100, "$gc_mediana"]}, 2]}}},
        {"$sort": {"gc_promedio": -1}},
    ]


pipeline_q6 = [{"$facet": {
    "orden": gc_por("$classification.te_order"),
    "reino": gc_por("$kingdom"),
    "reino_orden": gc_por({"reino": "$kingdom", "orden": "$classification.te_order"}),
}}]
q6 = {nivel: a_tabla(filas) for nivel, filas in next(tes.aggregate(pipeline_q6)).items()}
display(q6["reino"].rename(columns={"_id": "reino"}))
q6["orden"].rename(columns={"_id": "orden"})

In [ ]:
# % GC promedio de cada orden en cada reino
q6["reino_orden"].pivot_table(index="orden", columns="reino", values="gc_promedio")

**Interpretación:** _pendiente; se escribe con los resultados de la base completa._

### Consulta 7

**Pregunta:** ¿Qué TEs son los más largos y los más cortos de cada reino? Mostrar su nombre, clasificación y especie.

Se usa `$setWindowFields` con `$rank` particionado por reino: el rango 1 en orden descendente es el más
largo y el rango 1 en orden ascendente es el más corto. A diferencia de un `$sort` + `$group` con
`$first`, `$rank` conserva los empates (varios TEs con la misma longitud extrema). Antes se proyectan solo
los campos necesarios para que el ordenamiento use menos memoria.

In [ ]:
pipeline_q7 = [
    {"$project": {"_id": 0, "pdb_id": 1, "kingdom": 1, "length": 1, "species_name": 1, "classification": 1}},
    {"$setWindowFields": {"partitionBy": "$kingdom", "sortBy": {"length": -1},
                          "output": {"rango_largo": {"$rank": {}}}}},
    {"$setWindowFields": {"partitionBy": "$kingdom", "sortBy": {"length": 1},
                          "output": {"rango_corto": {"$rank": {}}}}},
    {"$match": {"$or": [{"rango_largo": 1}, {"rango_corto": 1}]}},
    {"$project": {"reino": "$kingdom",
                  "extremo": {"$cond": [{"$eq": ["$rango_largo", 1]}, "más largo", "más corto"]},
                  "longitud": "$length", "pdb_id": 1,
                  "clase": "$classification.te_class", "orden": "$classification.te_order",
                  "superfamilia": "$classification.te_superfamily", "especie": "$species_name"}},
    {"$sort": {"reino": 1, "extremo": -1, "pdb_id": 1}},
]
q7 = pd.DataFrame(list(tes.aggregate(pipeline_q7, allowDiskUse=True)))
print("TEs por reino y extremo (más de 1 indica empate):")
print(q7.groupby(["reino", "extremo"]).size().to_string())
q7.groupby(["reino", "extremo"], sort=False).head(5)   # máximo 5 por grupo si hay muchos empates

**Interpretación:** _pendiente; se escribe con los resultados de la base completa._

## Nombres y búsqueda de secuencias

### Consulta 8

**Pregunta:** Buscar una secuencia por su nombre (por ejemplo, `PDB00000001`) y mostrar su clasificación,
especie y taxonomía completa.

El `$match` por `pdb_id` usa el índice único `{pdb_id: 1}` y el `$lookup` por `species_id` usa el `_id` de
`especies`, así que la consulta lee un documento de cada colección. La taxonomía completa se trae de
`especies` (no está copiada en `tes`, decisión 1). La función recibe el nombre como parámetro.

In [ ]:
def buscar_te(pdb_id):
    pipeline = [
        {"$match": {"pdb_id": pdb_id}},
        {"$lookup": {"from": "especies", "localField": "species_id", "foreignField": "_id", "as": "especie"}},
        {"$unwind": "$especie"},
        {"$project": {"_id": 0, "pdb_id": 1, "clasificacion": "$classification",
                      "especie_fasta": "$species_name", "especie_csv": "$especie.species",
                      "taxonomia": "$especie.taxonomy", "fuente": "$especie.source",
                      "fuente_2": "$especie.source_2", "notas": "$especie.notes",
                      "longitud": "$length",
                      "gc_pct": {"$round": [{"$multiply": [100, "$gc_content"]}, 2]},
                      "n_pct": {"$round": [{"$multiply": [100, "$n_percent"]}, 2]}}},
    ]
    return next(tes.aggregate(pipeline), None)


print(json.dumps(buscar_te(PDB_ID), indent=2, ensure_ascii=False))

Un identificador con el formato de las bibliotecas de Dfam (no `PDB` + 8 dígitos) se busca igual, porque
`pdb_id` guarda el nombre original:

In [ ]:
otro = tes.find_one({"pdb_id": {"$not": {"$regex": "^PDB"}}}, {"pdb_id": 1})
print(json.dumps(buscar_te(otro["pdb_id"]), indent=2, ensure_ascii=False) if otro else "No hay identificadores sin formato PDB")

In [ ]:
# Plan de la búsqueda: debe ser IXSCAN sobre pdb_id_1, con 1 clave y 1 documento examinados
resumen_explain({"find": "tes", "filter": {"pdb_id": PDB_ID}})

**Interpretación:** _pendiente; se escribe con los resultados de la base completa._

### Consulta 9

**Pregunta:** Listar los nombres de todas las secuencias de una superfamilia en una especie dada.

El nombre de la especie se resuelve con `resolver_especie` y se pasa al nombre que guarda `tes.species_name`
(el del FASTA), para que el filtro use el índice compuesto `{species_name: 1, classification.te_superfamily: 1}`.
Si la superfamilia no existe en esa especie, la función lo dice y lista las que sí tiene.

In [ ]:
def tes_de_superfamilia(nombre_especie, superfamilia):
    esp = resolver_especie(nombre_especie)
    nombre_fasta = tes.find_one({"species_id": esp["_id"]}, {"species_name": 1})["species_name"]
    filtro = {"species_name": nombre_fasta, "classification.te_superfamily": superfamilia.strip().upper()}
    nombres = [d["pdb_id"] for d in tes.find(filtro, {"_id": 0, "pdb_id": 1}).sort("pdb_id", 1)]
    if not nombres:
        disponibles = sorted(tes.distinct("classification.te_superfamily", {"species_name": nombre_fasta}))
        print(f"{nombre_fasta} no tiene TEs de {superfamilia}. Superfamilias disponibles: {disponibles}")
    return nombres, filtro


nombres, filtro = tes_de_superfamilia(ESPECIE, SUPERFAMILIA)
print(f"{len(nombres)} TEs de {SUPERFAMILIA} en {filtro['species_name']}:")
print(nombres)

In [ ]:
resumen_explain({"find": "tes", "filter": filtro, "projection": {"_id": 0, "pdb_id": 1}})

Con una especie cuyo nombre difiere entre el CSV y el FASTA (`subsp` frente a `subsp.`), usando su
superfamilia más frecuente:

In [ ]:
ejemplo = "Beta vulgaris subsp vulgaris"   # nombre del CSV
esp = resolver_especie(ejemplo)
sf = next(tes.aggregate([
    {"$match": {"species_id": esp["_id"]}},
    {"$sortByCount": "$classification.te_superfamily"},
    {"$limit": 1},
]), {}).get("_id")
if sf:
    nombres_b, filtro_b = tes_de_superfamilia(ejemplo, sf)
    print(f"{len(nombres_b)} TEs de {sf} en {filtro_b['species_name']}; primeros: {nombres_b[:10]}")

**Interpretación:** _pendiente; se escribe con los resultados de la base completa._

## Relaciones y calidad

### Consulta 12 (`$lookup`)

**Pregunta:** Unir las secuencias con la taxonomía de su especie.

**a) De la secuencia a su especie.** Los primeros TEs de la especie del parámetro, cada uno con la
taxonomía completa traída de `especies`.

In [ ]:
esp = resolver_especie(ESPECIE)
pipeline_q12a = [
    {"$match": {"species_id": esp["_id"]}},
    {"$sort": {"pdb_id": 1}},
    {"$limit": 5},
    {"$lookup": {"from": "especies", "localField": "species_id", "foreignField": "_id", "as": "especie"}},
    {"$unwind": "$especie"},
    {"$project": {"_id": 0, "pdb_id": 1, "te_orden": "$classification.te_order",
                  "te_superfamilia": "$classification.te_superfamily",
                  "reino": "$especie.taxonomy.kingdom", "phylum": "$especie.taxonomy.phylum",
                  "clase_tax": "$especie.taxonomy.class", "orden_tax": "$especie.taxonomy.order",
                  "familia": "$especie.taxonomy.family", "especie": "$especie.species"}},
]
pd.DataFrame(list(tes.aggregate(pipeline_q12a)))

**b) Agregación con `$lookup`: ¿cuántos TEs hay por clase taxonómica de la especie en cada reino?**
La clase taxonómica (`taxonomy.class`) no está copiada en `tes`, así que hace falta unir las colecciones.
Se comparan dos formas de escribirlo:

- **Directa:** `$lookup` por cada uno de los 239.946 TEs y luego `$group`.
- **Agrupando primero:** contar los TEs por `species_id` (2.790 grupos) y hacer el `$lookup` solo una vez por
  especie. Da el mismo resultado con unas 86 veces menos búsquedas en `especies`.

In [ ]:
agrupar_por_clase_tax = [
    {"$group": {"_id": {"reino": "$esp.taxonomy.kingdom", "clase_tax": {"$ifNull": ["$esp.taxonomy.class", "(sin dato)"]}},
                "n_especies": {"$addToSet": "$esp._id"}, "n_tes": {"$sum": "$n"}}},
    {"$project": {"_id": 0, "reino": "$_id.reino", "clase_tax": "$_id.clase_tax",
                  "n_especies": {"$size": "$n_especies"}, "n_tes": 1}},
    {"$sort": {"reino": 1, "n_tes": -1, "clase_tax": 1}},
]
lookup_especie = {"$lookup": {"from": "especies", "localField": "species_id", "foreignField": "_id", "as": "esp"}}

pipeline_q12_directa = [
    lookup_especie, {"$unwind": "$esp"}, {"$set": {"n": 1}},
] + agrupar_por_clase_tax

pipeline_q12_agrupada = [
    {"$group": {"_id": "$species_id", "n": {"$sum": 1}}},
    {"$set": {"species_id": "$_id"}},
    lookup_especie, {"$unwind": "$esp"},
] + agrupar_por_clase_tax

directa, t_directa = medir(lambda: list(tes.aggregate(pipeline_q12_directa, allowDiskUse=True)))
agrupada, t_agrupada = medir(lambda: list(tes.aggregate(pipeline_q12_agrupada, allowDiskUse=True)))
assert directa == agrupada, "Las dos versiones deben dar lo mismo"
print(f"Directa: {t_directa:.2f} s | Agrupando primero: {t_agrupada:.2f} s | mismo resultado")
q12 = pd.DataFrame(agrupada)
q12["tes_por_especie"] = (q12["n_tes"] / q12["n_especies"]).round(1)
q12

**c) De la especie a sus secuencias (sentido inverso).** Desde `especies`, un `$lookup` a `tes` por
`species_id` (usa el índice `{species_id: 1}`) con una sub-pipeline que cuenta los TEs por orden.

In [ ]:
pipeline_q12c = [
    {"$match": {"_id": esp["_id"]}},
    {"$lookup": {"from": "tes", "localField": "_id", "foreignField": "species_id", "as": "tes_por_orden",
                 "pipeline": [{"$sortByCount": "$classification.te_order"}]}},
    {"$project": {"_id": 0, "species": 1, "taxonomy": 1, "source": 1, "tes_por_orden": 1}},
]
print(json.dumps(next(especies.aggregate(pipeline_q12c)), indent=2, ensure_ascii=False))

**Interpretación:** _pendiente; se escribe con los resultados de la base completa._

### Consulta 13 (comparación por Source)

**Pregunta:** Agrupando las especies según su `Source` (Dfam o RM2+MCH): ¿cuántas especies y cuántas
secuencias hay en cada grupo? ¿Cuál es el promedio de secuencias por especie? ¿Cuál es la longitud promedio
de los TEs?

`source` está copiado en `tes`, así que no hace falta `$lookup`. Primero se agrupa por (`source`,
`species_id`) para tener los TEs de cada especie y luego por `source`. El promedio de longitud se calcula
sobre todos los TEs del grupo (suma de longitudes / número de TEs). El número de especies se contrasta con
la colección `especies`.

In [ ]:
pipeline_q13 = [
    {"$group": {"_id": {"source": "$source", "species_id": "$species_id"},
                "n_tes": {"$sum": 1}, "suma_long": {"$sum": "$length"}}},
    {"$group": {"_id": "$_id.source", "n_especies": {"$sum": 1}, "n_tes": {"$sum": "$n_tes"},
                "suma_long": {"$sum": "$suma_long"},
                "mediana_tes_por_especie": {"$median": {"input": "$n_tes", "method": "approximate"}},
                "min_tes_por_especie": {"$min": "$n_tes"}, "max_tes_por_especie": {"$max": "$n_tes"}}},
    {"$project": {"_id": 0, "source": "$_id", "n_especies": 1, "n_tes": 1,
                  "promedio_tes_por_especie": {"$round": [{"$divide": ["$n_tes", "$n_especies"]}, 1]},
                  "mediana_tes_por_especie": 1, "min_tes_por_especie": 1, "max_tes_por_especie": 1,
                  "longitud_promedio": {"$round": [{"$divide": ["$suma_long", "$n_tes"]}, 1]}}},
    {"$sort": {"source": 1}},
]
q13 = pd.DataFrame(list(tes.aggregate(pipeline_q13)))[
    ["source", "n_especies", "n_tes", "promedio_tes_por_especie", "mediana_tes_por_especie",
     "min_tes_por_especie", "max_tes_por_especie", "longitud_promedio"]]

especies_por_source = {d["_id"]: d["n"] for d in especies.aggregate([{"$group": {"_id": "$source", "n": {"$sum": 1}}}])}
q13["n_especies_en_coleccion_especies"] = q13["source"].map(especies_por_source)
q13

Para interpretar la diferencia, la longitud mediana y la composición por orden de cada grupo:

In [ ]:
pipeline_q13_detalle = [
    {"$facet": {
        "longitud": [{"$group": {"_id": "$source",
                                 "mediana": {"$median": {"input": "$length", "method": "approximate"}},
                                 "p90": {"$percentile": {"input": "$length", "p": [0.9], "method": "approximate"}}}},
                      {"$set": {"p90": {"$first": "$p90"}}}],
        "composicion": composicion_por("$source", "source"),
    }},
]
q13_det = next(tes.aggregate(pipeline_q13_detalle))
display(a_tabla(q13_det["longitud"], id_como="source"))
a_tabla(q13_det["composicion"]).pivot_table(index="orden", columns="source", values="porcentaje", fill_value=0)

**Interpretación:** _pendiente; se escribe con los resultados de la base completa._

### Consulta 14 (control de calidad)

**Pregunta:** ¿Cuántos TEs no están clasificados hasta superfamilia, en qué órdenes y de qué fuente vienen?

En el FASTA, cuando la superfamilia repite el nombre del orden (`CLASSI/LTR/LTR`) puede significar dos
cosas: que el orden tiene una única superfamilia homónima (correcto) o que el TE solo se clasificó hasta
orden. La exploración (`MODELO.md`, Calidad de datos) separó los casos en tres grupos, que aquí se
calculan con `$expr` y `$switch`:

| Grupo | Órdenes |
| --- | --- |
| Correcto (superfamilia única homónima) | HELITRON, CRYPTON, MAVERICK, PLE |
| Clasificado solo hasta orden | TIR, LINE, LTR, SINE |
| Ambiguo | MITE, DIRS, RETROPOSON |

In [ ]:
CORRECTOS = ["HELITRON", "CRYPTON", "MAVERICK", "PLE"]
SOLO_ORDEN = ["TIR", "LINE", "LTR", "SINE"]
AMBIGUOS = ["MITE", "DIRS", "RETROPOSON"]

pipeline_q14 = [
    {"$group": {"_id": "$classification.te_order", "total_orden": {"$sum": 1},
                "sf_igual_orden": {"$sum": {"$cond": [
                    {"$eq": ["$classification.te_superfamily", "$classification.te_order"]}, 1, 0]}}}},
    {"$match": {"sf_igual_orden": {"$gt": 0}}},
    {"$project": {"_id": 0, "orden": "$_id", "total_orden": 1, "sf_igual_orden": 1,
                  "pct_del_orden": {"$round": [{"$multiply": [100, {"$divide": ["$sf_igual_orden", "$total_orden"]}]}, 1]},
                  "grupo": {"$switch": {"branches": [
                      {"case": {"$in": ["$_id", CORRECTOS]}, "then": "correcto"},
                      {"case": {"$in": ["$_id", SOLO_ORDEN]}, "then": "solo hasta orden"},
                      {"case": {"$in": ["$_id", AMBIGUOS]}, "then": "ambiguo"}],
                      "default": "sin revisar"}}}},
    {"$sort": {"grupo": 1, "sf_igual_orden": -1}},
]
q14 = pd.DataFrame(list(tes.aggregate(pipeline_q14)))[["grupo", "orden", "sf_igual_orden", "total_orden", "pct_del_orden"]]
display(q14)
q14.groupby("grupo")["sf_igual_orden"].sum().rename("n_tes").to_frame().assign(
    porcentaje_de_la_base=lambda d: (100 * d["n_tes"] / tes.estimated_document_count()).round(2))

Los clasificados solo hasta orden, por fuente y reino: muestra si el problema se concentra en las bibliotecas
generadas automáticamente (RM2+MCH) o en las curadas (Dfam).

In [ ]:
pipeline_q14_fuente = [
    {"$match": {"classification.te_order": {"$in": SOLO_ORDEN},
                "$expr": {"$eq": ["$classification.te_superfamily", "$classification.te_order"]}}},
    {"$group": {"_id": {"fuente": "$source", "reino": "$kingdom"}, "n_tes": {"$sum": 1},
                "especies": {"$addToSet": "$species_id"}}},
    {"$project": {"n_tes": 1, "n_especies": {"$size": "$especies"}}},
    {"$sort": {"n_tes": -1}},
]
q14_fuente = a_tabla(tes.aggregate(pipeline_q14_fuente))[["fuente", "reino", "n_tes", "n_especies"]]
# Proporción sobre el total de TEs de esos órdenes en cada fuente
totales = a_tabla(tes.aggregate([
    {"$match": {"classification.te_order": {"$in": SOLO_ORDEN}}},
    {"$group": {"_id": {"fuente": "$source", "reino": "$kingdom"}, "n_total": {"$sum": 1}}},
]))
q14_fuente = q14_fuente.merge(totales, on=["fuente", "reino"])
q14_fuente["pct_sin_superfamilia"] = (100 * q14_fuente["n_tes"] / q14_fuente["n_total"]).round(1)
q14_fuente

Controles complementarios de integridad, que el modelo no garantiza por sí solo (MongoDB no tiene llaves
foráneas):

- TEs cuyo `species_id` no existe en `especies` (se espera 0).
- TEs de entradas que no son especies (`is_species: false`).
- TEs que no tienen exactamente 6 documentos en `kmers` (se espera 0).
- Secuencias con N: cuántas y con qué porcentaje.

In [ ]:
huerfanos = next(tes.aggregate([
    {"$group": {"_id": "$species_id", "n": {"$sum": 1}}},
    {"$lookup": {"from": "especies", "localField": "_id", "foreignField": "_id", "as": "e"}},
    {"$match": {"e": {"$size": 0}}},
    {"$group": {"_id": None, "especies": {"$sum": 1}, "tes": {"$sum": "$n"}}},
]), {"especies": 0, "tes": 0})
print(f"TEs sin especie en 'especies': {huerfanos['tes']} (de {huerfanos['especies']} species_id)")

no_especie = pd.DataFrame(list(tes.aggregate([
    {"$group": {"_id": "$species_id", "nombre": {"$first": "$species_name"}, "n_tes": {"$sum": 1}}},
    {"$lookup": {"from": "especies", "localField": "_id", "foreignField": "_id", "as": "e"}},
    {"$match": {"e.is_species": False}},
    {"$project": {"_id": 0, "nombre": 1, "n_tes": 1}},
    {"$sort": {"n_tes": -1}},
])))
print(f"Entradas con is_species = false: {len(no_especie)}, con {no_especie['n_tes'].sum() if len(no_especie) else 0:,} TEs")
display(no_especie.head(10))

kmers_incompletos, t = medir(lambda: next(kmers.aggregate([
    {"$group": {"_id": "$te_id", "n": {"$sum": 1}}},
    {"$match": {"n": {"$ne": 6}}},
    {"$count": "n"},
], allowDiskUse=True), {"n": 0}))
print(f"TEs con un número de documentos en kmers distinto de 6: {kmers_incompletos['n']} ({t:.1f} s)")

pipeline_n = [
    {"$bucket": {"groupBy": "$n_percent", "boundaries": [0, 1e-12, 0.01, 0.05, 0.10, 0.50, 1.000001],
                 "output": {"n_tes": {"$sum": 1}}}},
]
etiquetas = {0: "sin N", 1e-12: "(0 %, 1 %)", 0.01: "[1 %, 5 %)", 0.05: "[5 %, 10 %)", 0.10: "[10 %, 50 %)", 0.50: "≥ 50 %"}
buckets_n = pd.DataFrame(list(tes.aggregate(pipeline_n)))
buckets_n["rango_de_N"] = buckets_n["_id"].map(etiquetas)
buckets_n["porcentaje"] = (100 * buckets_n["n_tes"] / buckets_n["n_tes"].sum()).round(2)
buckets_n[["rango_de_N", "n_tes", "porcentaje"]]

**Interpretación:** _pendiente; se escribe con los resultados de la base completa._

## K-mers

### Consulta 15

**Pregunta:** Recuperar el perfil de k-mers de una secuencia dada, para un k dado.

Es la Q1 de `06_benchmark_consultas.ipynb` (versión `coleccion`): un `find` en `kmers` por `pdb_id` y `k`,
que usa el índice `{pdb_id: 1, k: 1}` y lee un solo documento. Como `counts` guarda solo los k-mers
presentes (decisión 5), la función completa los ausentes con 0 y calcula la frecuencia como
`conteo / valid_windows`.

In [ ]:
from itertools import product


def perfil_kmers(pdb_id, k):
    doc = kmers.find_one({"pdb_id": pdb_id, "k": k}, {"_id": 0, "valid_windows": 1, "counts": 1})
    if doc is None:
        raise ValueError(f"No hay perfil de k = {k} para {pdb_id}")
    todos = ["".join(p) for p in product("ACGT", repeat=k)]
    perfil = pd.DataFrame({"kmer": todos, "conteo": [doc["counts"].get(km, 0) for km in todos]})
    perfil["frecuencia"] = perfil["conteo"] / doc["valid_windows"] if doc["valid_windows"] else 0.0
    assert perfil["conteo"].sum() == doc["valid_windows"], "La suma de conteos debe ser igual a valid_windows"
    return doc["valid_windows"], perfil


ventanas, perfil = perfil_kmers(PDB_ID, K)
print(f"{PDB_ID}, k = {K}: {ventanas:,} ventanas válidas, {int((perfil['conteo'] > 0).sum())} de {4 ** K} k-mers presentes")
perfil.sort_values("conteo", ascending=False).head(20).round(4)

In [ ]:
resumen_explain({"find": "kmers", "filter": {"pdb_id": PDB_ID, "k": K},
                 "projection": {"_id": 0, "valid_windows": 1, "counts": 1}})

Cobertura del perfil completo de la misma secuencia (sus 6 documentos): cuántos de los 4ᵏ k-mers posibles
aparecen para cada k. Muestra por qué el diccionario disperso ahorra espacio solo para k grandes.

In [ ]:
cobertura = pd.DataFrame([
    {"k": d["k"], "valid_windows": d["valid_windows"], "presentes": len(d["counts"]), "posibles": 4 ** d["k"]}
    for d in kmers.find({"pdb_id": PDB_ID}, {"_id": 0, "k": 1, "valid_windows": 1, "counts": 1}).sort("k", 1)
])
cobertura["pct_presentes"] = (100 * cobertura["presentes"] / cobertura["posibles"]).round(1)
cobertura

**Interpretación:** _pendiente; se escribe con los resultados de la base completa._

## Consultas propias

### Propia 1

**Pregunta:** ¿Cuántas superfamilias distintas tiene cada especie? ¿Depende del reino y de la fuente, o
solo del número de TEs?

Una especie con más TEs tiene más oportunidades de mostrar superfamilias distintas, así que además del
promedio por grupo se mide la correlación de Spearman entre el número de TEs y la diversidad.

In [ ]:
pipeline_p1 = [
    {"$group": {"_id": "$species_id", "reino": {"$first": "$kingdom"}, "fuente": {"$first": "$source"},
                "n_tes": {"$sum": 1},
                "superfamilias": {"$addToSet": {"o": "$classification.te_order", "s": "$classification.te_superfamily"}},
                "ordenes": {"$addToSet": "$classification.te_order"}}},
    {"$project": {"_id": 0, "reino": 1, "fuente": 1, "n_tes": 1,
                  "n_superfamilias": {"$size": "$superfamilias"}, "n_ordenes": {"$size": "$ordenes"}}},
]
p1 = pd.DataFrame(list(tes.aggregate(pipeline_p1)))
resumen_p1 = (p1.groupby(["reino", "fuente"])
              .agg(n_especies=("n_tes", "size"), tes_promedio=("n_tes", "mean"),
                   superfamilias_promedio=("n_superfamilias", "mean"),
                   superfamilias_max=("n_superfamilias", "max"), ordenes_promedio=("n_ordenes", "mean"))
              .round(1))
display(resumen_p1)
rho = p1["n_tes"].corr(p1["n_superfamilias"], method="spearman")
print(f"Correlación de Spearman entre número de TEs y número de superfamilias: {rho:.2f} (n = {len(p1):,} especies)")

**Interpretación:** _pendiente; se escribe con los resultados de la base completa._

### Propia 2

**Pregunta:** ¿Están empobrecidos en dinucleótidos CG (CpG) los TEs, y cambia según el orden y el reino?

En genomas con metilación del ADN las citosinas metiladas de los CpG mutan con frecuencia a T, así que las
secuencias que han estado metiladas tienen menos CG del esperado por su contenido de C y G. La medida
habitual es la razón observado/esperado:

$$\text{CpG}_{o/e} = \frac{f(CG)}{f(C)\,f(G)}$$

con f(C) y f(G) del perfil de k = 1 y f(CG) del de k = 2. Valores cercanos a 1 indican que no hay
empobrecimiento y valores bajos (por debajo de ~0,6) indican un empobrecimiento marcado.

La consulta usa la colección `kmers` con el índice `{k: 1, te_order: 1}` (lee solo k = 1 y 2), arma la razón
por TE agrupando por `te_id` y trae el reino con un `$lookup` a `tes`, porque en `kmers` solo se copió
`te_order` (decisión 4).

In [ ]:
pipeline_p2 = [
    {"$match": {"k": {"$in": [1, 2]}, "valid_windows": {"$gt": 0}}},
    {"$group": {"_id": "$te_id", "orden": {"$first": "$te_order"},
                "fC": {"$max": {"$cond": [{"$eq": ["$k", 1]}, {"$divide": [{"$ifNull": ["$counts.C", 0]}, "$valid_windows"]}, None]}},
                "fG": {"$max": {"$cond": [{"$eq": ["$k", 1]}, {"$divide": [{"$ifNull": ["$counts.G", 0]}, "$valid_windows"]}, None]}},
                "fCG": {"$max": {"$cond": [{"$eq": ["$k", 2]}, {"$divide": [{"$ifNull": ["$counts.CG", 0]}, "$valid_windows"]}, None]}}}},
    {"$match": {"fC": {"$gt": 0}, "fG": {"$gt": 0}, "fCG": {"$ne": None}}},
    {"$set": {"cpg_oe": {"$divide": ["$fCG", {"$multiply": ["$fC", "$fG"]}]}}},
    {"$lookup": {"from": "tes", "localField": "_id", "foreignField": "_id", "as": "te",
                 "pipeline": [{"$project": {"_id": 0, "kingdom": 1}}]}},
    {"$group": {"_id": {"reino": {"$first": "$te.kingdom"}, "orden": "$orden"}, "n_tes": {"$sum": 1},
                "cpg_oe_promedio": {"$avg": "$cpg_oe"},
                "cpg_oe_mediana": {"$median": {"input": "$cpg_oe", "method": "approximate"}}}},
    {"$set": {"cpg_oe_promedio": {"$round": ["$cpg_oe_promedio", 3]}, "cpg_oe_mediana": {"$round": ["$cpg_oe_mediana", 3]}}},
]
filas_p2, t = medir(lambda: list(kmers.aggregate(pipeline_p2, allowDiskUse=True)))
p2 = a_tabla(filas_p2)
print(f"{p2['n_tes'].sum():,} TEs con razón CpG o/e | {t:.1f} s")

# Promedio por reino (todos los órdenes, ponderado por número de TEs) y mediana por orden y reino
por_reino = (p2.assign(suma=p2["cpg_oe_promedio"] * p2["n_tes"]).groupby("reino")[["n_tes", "suma"]].sum()
             .assign(cpg_oe_promedio=lambda d: (d["suma"] / d["n_tes"]).round(3)).drop(columns="suma"))
display(por_reino)
p2.pivot_table(index="orden", columns="reino", values="cpg_oe_mediana")

**Interpretación:** _pendiente; se escribe con los resultados de la base completa._

## Resumen de índices usados

| Consulta | Operación principal | Índice |
| --- | --- | --- |
| 1, 3, 5, 6, 7, 13, 14 | `$group` / `$setWindowFields` sobre toda `tes` | Ninguno (COLLSCAN, se leen todos los TEs) |
| 2 (una especie), 12a | `$match` por `species_id` | `{species_id: 1}` |
| 4, 12b, 14 (integridad) | `$lookup` a `especies` por `_id` | `_id` de `especies` |
| 8 | `$match` por `pdb_id` + `$lookup` | `{pdb_id: 1}` único + `_id` de `especies` |
| 9 | `find` por especie y superfamilia | `{species_name: 1, classification.te_superfamily: 1}` |
| 12c | `$lookup` de `especies` a `tes` | `{species_id: 1}` |
| 15 | `find` por `pdb_id` y `k` | `{pdb_id: 1, k: 1}` |
| Propia 2 | `$match` por `k` en `kmers` + `$lookup` a `tes` | `{k: 1, te_order: 1}` + `_id` de `tes` |